In [1]:
# Loading libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import joblib


In [2]:
# Load datasets
df_original = pd.read_csv('../Datasets/creditcard_preprocessed.csv')
df_augmented = pd.read_csv('../Datasets/creditcard_augmented.csv')


In [3]:
# Separate features and target
X_orig = df_original.drop('Class', axis=1)
y_orig = df_original['Class']

X_aug = df_augmented.drop('Class', axis=1)
y_aug = df_augmented['Class']

In [4]:
# Train-test split (stratified)
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(X_orig, y_orig, test_size=0.3, stratify=y_orig, random_state=42)
X_train_aug, X_test_aug, y_train_aug, y_test_aug = train_test_split(X_aug, y_aug, test_size=0.3, stratify=y_aug, random_state=42)


In [5]:
# Model initialization
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

def train_and_eval(X_train, y_train, X_test, y_test):
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probas = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, probas)
        report = classification_report(y_test, preds, output_dict=True)
        results[name] = {
            "model": model,
            "roc_auc": roc_auc,
            "classification_report": report,
            "preds": preds,
            "probas": probas
        }
    return results

In [6]:
# Train and evaluate models on original data
results_orig = train_and_eval(X_train_orig, y_train_orig, X_test_orig, y_test_orig)

# Train and evaluate models on augmented data
results_aug = train_and_eval(X_train_aug, y_train_aug, X_test_aug, y_test_aug)

In [10]:
# Compare ROC-AUC Scores
print("ROC-AUC Scores Comparison")
for name in models.keys():
    print(f"{name}: Original = {results_orig[name]['roc_auc']:.4f}, Augmented = {results_aug[name]['roc_auc']:.4f}")

ROC-AUC Scores Comparison
Logistic Regression: Original = 0.9573, Augmented = 0.8144
Random Forest: Original = 0.9630, Augmented = 0.9922
XGBoost: Original = 0.9390, Augmented = 0.9933


In [7]:
# Compare metrics for different models for both real and augmented datasets
model_names = list(models.keys())
roc_auc_orig = [results_orig[name]['roc_auc'] for name in model_names]
roc_auc_aug = [results_aug[name]['roc_auc'] for name in model_names]
metric_table = pd.DataFrame({
    "Model": model_names,
    "ROC-AUC (Original)": roc_auc_orig,
    "ROC-AUC (Augmented)": roc_auc_aug,
    "F1-Score (Original)": [results_orig[name]['classification_report']['1']['f1-score'] for name in model_names],
    "F1-Score (Augmented)": [results_aug[name]['classification_report']['1']['f1-score'] for name in model_names],
    "Recall (Original)": [results_orig[name]['classification_report']['1']['recall'] for name in model_names],
    "Recall (Augmented)": [results_aug[name]['classification_report']['1']['recall'] for name in model_names],
    "Precision (Original)": [results_orig[name]['classification_report']['1']['precision'] for name in model_names],
    "Precision (Augmented)": [results_aug[name]['classification_report']['1']['precision'] for name in model_names],
})

In [8]:
metric_table

,Model,ROC-AUC (Original),ROC-AUC (Augmented),F1-Score (Original),F1-Score (Augmented),Recall (Original),Recall (Augmented),Precision (Original),Precision (Augmented)
0,Logistic Regression,0.955981,0.777248,0.716535,0.438292,0.614865,0.283374,0.858491,0.966874
1,Random Forest,0.930739,0.990540,0.845283,0.639706,0.756757,0.475121,0.957265,0.978750
2,XGBoost,0.928599,0.992197,0.838951,0.870239,0.756757,0.783374,0.941176,0.978772


As you can see from this the best model is XGBoost. In every metric, it gives good results and imporved results for augmeneted dataset.

In [10]:
# Save the best model from augmented results which is XGBoost
best_model = results_aug['XGBoost']['model']
joblib.dump(best_model, '../Model/credit_card_fraud_detection_model.pkl')

['../Model/credit_card_fraud_detection_model.pkl']

In [13]:
# Stroing varaibles which can be used further in other notebooks
%store results_orig
%store results_aug

Stored 'results_orig' (dict)
Stored 'results_aug' (dict)


In [15]:
%store X_test_aug y_test_aug

Stored 'X_test_aug' (DataFrame)
Stored 'y_test_aug' (Series)
